## Create a new environment in bioconda (miniconda)

In [ ]:
!conda create -n 16s-nanopore -c bioconda -c conda-forge nanoplot cutadapt chopper kma emu osfclient -y
!conda activate 16s-nanopore

In [ ]:
# Sanity check using 'conda run' to point to the correct environment
!conda run -n 16s-nanopore NanoPlot --version
!conda run -n 16s-nanopore cutadapt --version
!conda run -n 16s-nanopore chopper --version
!conda run -n 16s-nanopore emu --version
!conda run -n 16s-nanopore osf -V
!conda run -n 16s-nanopore kma -v

## Renamed

In [ ]:
import os
import shutil
import pandas as pd

# 1. Paths
folder = '/home/marcos/colaboracao/paulo_de_melo/RICHARDT-GAMA-LANDGRAF_results/RICHARDT-GAMA-LANDGRAF_results/fastqs/16s'
csv_file = '/home/marcos/colaboracao/paulo_de_melo/RICHARDT-GAMA-LANDGRAF_results/RICHARDT-GAMA-LANDGRAF_results/fastqs/meta_date_16s.csv'
renamed_folder = os.path.join(folder, 'renamed')  # raw files stay untouched
os.makedirs(renamed_folder, exist_ok=True)

# 2. Read metadata (comma is the expected separator for a .csv)
df = pd.read_csv(csv_file, sep=',')
assert {'Name', 'sample_title'}.issubset(df.columns), f"Missing columns, found: {df.columns.tolist()}"

dupes = df['sample_title'][df['sample_title'].duplicated()].tolist()
if dupes:
    raise ValueError(f"Duplicate sample_title values would overwrite files: {dupes}")

log = []

# 3. Copy files under the new name, preserving the original extension
for _, row in df.iterrows():
    old_name = str(row['Name']).strip()
    sample_title = str(row['sample_title']).strip()
    old_path = os.path.join(folder, old_name)

    if not os.path.exists(old_path):
        print(f"[ERROR] File not found: {old_name}")
        continue

    if old_name.endswith('.fastq.gz'):
        ext = '.fastq.gz'
    elif old_name.endswith('.fastq'):
        ext = '.fastq'
    else:
        ext = os.path.splitext(old_name)[1]

    new_name = f"{sample_title}{ext}"
    new_path = os.path.join(renamed_folder, new_name)

    shutil.copy2(old_path, new_path)
    log.append({'old_name': old_name, 'new_name': new_name})
    print(f"[SUCCESS] Copied: {old_name} -> {new_name}")

pd.DataFrame(log).to_csv(os.path.join(renamed_folder, 'rename_manifest.csv'), index=False)
print("Done!")

## Library and directory preparation

In [ ]:
import os
import subprocess

# 1. Paths configuration (Updating based on your previous paths)
renamed_folder = '/home/marcos/colaboracao/paulo_de_melo/RICHARDT-GAMA-LANDGRAF_results/RICHARDT-GAMA-LANDGRAF_results/fastqs/16s/renamed'
base_out = '/home/marcos/colaboracao/paulo_de_melo/RICHARDT-GAMA-LANDGRAF_results/RICHARDT-GAMA-LANDGRAF_results/downstream'

# Define specific output folders for each step
qc_dir = os.path.join(base_out, '1_qc_raw')
trimmed_dir = os.path.join(base_out, '2_trimmed')
filtered_dir = os.path.join(base_out, '3_filtered')
emu_dir = os.path.join(base_out, '4_emu_taxa')

# Create directories if they don't exist
for directory in [qc_dir, trimmed_dir, filtered_dir, emu_dir]:
    os.makedirs(directory, exist_ok=True)

# 2. Get list of renamed files
fastq_files = [f for f in os.listdir(renamed_folder) if f.endswith('.fastq') or f.endswith('.fastq.gz')]
print(f"Found {len(fastq_files)} fastq files to process.")

## Quality control of the reads

In [ ]:
from pathlib import Path
import subprocess

env = "16s-nanopore"
threads = "4"

renamed_folder = Path("/home/marcos/colaboracao/paulo_de_melo/RICHARDT-GAMA-LANDGRAF_results/RICHARDT-GAMA-LANDGRAF_results/fastqs/16s/renamed")
qc_dir = Path("/home/marcos/colaboracao/paulo_de_melo/RICHARDT-GAMA-LANDGRAF_results/RICHARDT-GAMA-LANDGRAF_results/downstream/1_qc_raw")
qc_dir.mkdir(parents=True, exist_ok=True)

# Removido o multiqc da instalação
subprocess.run(
    f"conda install -n {env} -c bioconda -c conda-forge nanocomp -y",
    shell=True,
    check=True
)

fastq_files = sorted(
    file for file in renamed_folder.iterdir()
    if file.name.endswith((".fastq", ".fastq.gz"))
)

print(f"Starting QC for {len(fastq_files)} samples...")

for file in fastq_files:
    sample = file.name.split(".fastq")[0]
    print(f"Running NanoPlot (with dot plots): {sample}")

    subprocess.run([
        "conda", "run", "-n", env, "NanoPlot",
        "-t", threads, "--fastq", str(file),
        "-o", str(qc_dir / sample),
        "--plots", "dot"
    ], check=True)

nanocomp_out = qc_dir / "NanoComp_Report"
nanocomp_out.mkdir(exist_ok=True)

print("Generating NanoComp report...")
subprocess.run([
    "conda", "run", "-n", env, "NanoComp", "-t", threads,
    "--fastq", *map(str, fastq_files),
    "--names", *(file.name.split(".fastq")[0] for file in fastq_files),
    "-o", str(nanocomp_out)
], check=True)

print("\nQuality control completed!")
print(f"NanoComp report: {nanocomp_out}")

For decision trimmer, use the LengthvsQualityScatterPlot_kde, which is a tool that generates a scatter plot of read lengths versus quality scores. This can help identify any issues with the sequencing data, such as low-quality reads or unexpected length distributions.

## Filter reads by length with chopper

Trim the reads to a specific length range using chopper. This step is important to ensure that only reads of the desired length are retained for downstream analysis. The command will filter the reads based on the specified minimum and maximum length thresholds.

In [ ]:
# 5. Run Chopper for quality and length filtering
import os
import subprocess

renamed_folder = Path("/home/marcos/colaboracao/paulo_de_melo/RICHARDT-GAMA-LANDGRAF_results/RICHARDT-GAMA-LANDGRAF_results/fastqs/16s/renamed")
filtered_dir = Path("/home/marcos/colaboracao/paulo_de_melo/RICHARDT-GAMA-LANDGRAF_results/RICHARDT-GAMA-LANDGRAF_results/downstream/3_filtered")
filtered_dir.mkdir(parents=True, exist_ok=True)

fastq_files = [f for f in os.listdir(renamed_folder) if f.endswith('.fastq') or f.endswith('.fastq.gz')]

failed_samples = []

for file in fastq_files:
    sample_name = file.split('.fastq')[0]
    input_path = os.path.join(trimmed_dir, file)
    output_path = os.path.join(filtered_dir, f"{sample_name}_filtered.fastq")
    
    bash_cmd = (
        f"conda run -n 16s-nanopore bash -c "
        f"\"zcat -f '{input_path}' | "
        f"chopper "
        f"-q 10 "
        f"--minlength 1000 " #Is the minimum length of the reads to keep. Reads shorter than this will be discarded.
        f"--maxlength 1700 " #Is the maximum length of the reads to keep. Reads longer than this will be discarded.
        f"> '{output_path}'\""
    )
    
    print(f"Filtering {sample_name}...")
    
    try:
        # check=True Makes subprocess.run raise an exception if the command fails
        subprocess.run(bash_cmd, shell=True, check=True)
    except subprocess.CalledProcessError as e:
        print(f"\n[ERROR] {sample_name}!")
        print("Skipping to the next sample...\n")
        failed_samples.append(sample_name)
        
        if os.path.exists(output_path):
            os.remove(output_path)

print("\n--- Filtered end ---")
if failed_samples:
    print(f"Failed samples: {failed_samples}")
else:
    print("All samples were processed successfully!")

Counts reads before and after the quality control

In [ ]:
import os
import subprocess
from pathlib import Path

# 1. Define the input and output folders
renamed_folder = Path("/home/marcos/colaboracao/paulo_de_melo/RICHARDT-GAMA-LANDGRAF_results/RICHARDT-GAMA-LANDGRAF_results/fastqs/16s/renamed")
filtered_dir = Path("/home/marcos/colaboracao/paulo_de_melo/RICHARDT-GAMA-LANDGRAF_results/RICHARDT-GAMA-LANDGRAF_results/downstream/3_filtered")

# 2. Get the list of original fastq files
fastq_files = [f for f in os.listdir(renamed_folder) if f.endswith('.fastq') or f.endswith('.fastq.gz')]

print("\n--- Read Count Summary ---")

def get_read_count(filepath):
    """Returns the number of reads in a fastq or fastq.gz file."""
    if not os.path.exists(filepath):
        return 0
    try:
        # zcat -f safely reads both compressed and uncompressed files
        cmd = f"zcat -f '{filepath}' | wc -l"
        result = subprocess.check_output(cmd, shell=True, text=True)
        return int(result.strip()) // 4
    except Exception:
        return 0

# 3. Loop through the files to count reads before and after
for file in fastq_files:
    sample_name = file.split('.fastq')[0]
    
    input_path = os.path.join(renamed_folder, file) 
    output_path = os.path.join(filtered_dir, f"{sample_name}_filtered.fastq")
    
    reads_before = get_read_count(input_path)
    reads_after = get_read_count(output_path)
    
    if reads_before > 0:
        retention = (reads_after / reads_before) * 100
    else:
        retention = 0.0
        
    print(f"Sample: {sample_name}")
    print(f"  Before: {reads_before:,} reads")
    print(f"  After:  {reads_after:,} reads ({retention:.2f}% retained)")

## Classify the reads using the SILVA database

1. Download and extract the SILVA database for taxonomic classification.

Download the SILVA database (NR99 version) and extract it to the specified directory. This database will be used for taxonomic classification of the 16S rRNA gene sequences.

In [ ]:
%%bash
BASE_DIR="/home/marcos/colaboracao/paulo_de_melo/RICHARDT-GAMA-LANDGRAF_results/RICHARDT-GAMA-LANDGRAF_results/downstream"
DB_DIR="$BASE_DIR/silva_database"

mkdir -p $DB_DIR

# Download the SILVA database (NR99 version 138.2)
echo "Initiating download of the SILVA database"

wget -nc -O $DB_DIR/SILVA_138.2_SSURef_NR99_tax_silva.fasta.gz "https://www.arb-silva.de/fileadmin/silva_databases/release_138_2/Exports/SILVA_138.2_SSURef_NR99_tax_silva.fasta.gz"
gunzip -f $DB_DIR/SILVA_138.2_SSURef_NR99_tax_silva.fasta.gz

echo "Download and extraction completed successfully!"

2. Defining the reference database 

In [ ]:
%%bash
BASE_DIR="/home/marcos/colaboracao/paulo_de_melo/RICHARDT-GAMA-LANDGRAF_results/RICHARDT-GAMA-LANDGRAF_results/downstream"
DB_DIR="$BASE_DIR/silva_database"
KMA_DIR="$BASE_DIR/kma_silva_db"
mkdir -p $KMA_DIR

ORIGINAL_FASTA="$DB_DIR/SILVA_138.2_SSURef_NR99_tax_silva.fasta"
BACTERIA_FASTA="$DB_DIR/SILVA_16S_Bacteria.fasta"

echo "Initiating filtration of Eukaryotes and Archaea..."
awk '/^>/ {p = ($0 ~ /Bacteria;/)} p' $ORIGINAL_FASTA > $BACTERIA_FASTA
echo "Filtration completed! File BACTERIA_FASTA created."

3. Taxonomic extraction

In [ ]:
%%bash
BASE_DIR="/home/marcos/colaboracao/paulo_de_melo/RICHARDT-GAMA-LANDGRAF_results/RICHARDT-GAMA-LANDGRAF_results/downstream"
DB_DIR="$BASE_DIR/silva_database"
BACTERIA_FASTA="$DB_DIR/SILVA_16S_Bacteria.fasta"

echo "Extracting taxonomy for CSV..."
grep "^>" $BACTERIA_FASTA > $DB_DIR/headers_silva.txt
awk -F' ' '{print $1 "," substr($0, index($0,$2))}' $DB_DIR/headers_silva.txt > $DB_DIR/split_headers.csv
sed -i 's/;/,/g' $DB_DIR/split_headers.csv 
echo "id,Kingdom,Phylum,Class,Order,Family,Genus,Species" > $DB_DIR/silva_taxonomy.csv
sed 's/^>//' $DB_DIR/split_headers.csv >> $DB_DIR/silva_taxonomy.csv
echo "Taxonomy extracted successfully!"

4. Clear header FASTA

In [ ]:
%%bash
BASE_DIR="/home/marcos/colaboracao/paulo_de_melo/RICHARDT-GAMA-LANDGRAF_results/RICHARDT-GAMA-LANDGRAF_results/downstream"
DB_DIR="$BASE_DIR/silva_database"
BACTERIA_FASTA="$DB_DIR/SILVA_16S_Bacteria.fasta"

echo "Cleaning the FASTA file..."
sed -i 's/ .*//' $BACTERIA_FASTA
echo "Cleaned FASTA!"

5. Indexing in KMA

In [ ]:
%%bash
eval "$(conda shell.bash hook)"
conda activate 16s-nanopore
BASE_DIR="/home/marcos/colaboracao/paulo_de_melo/RICHARDT-GAMA-LANDGRAF_results/RICHARDT-GAMA-LANDGRAF_results/downstream"
DB_DIR="$BASE_DIR/silva_database"
KMA_DIR="$BASE_DIR/kma_silva_db"
BACTERIA_FASTA="$DB_DIR/SILVA_16S_Bacteria.fasta"

echo "Initiating KMA indexing of the SILVA database"
kma index -i $BACTERIA_FASTA -o $KMA_DIR/silva_db
echo "KMA indexing of the SILVA database completed successfully!"

6. KMA classification (SILVA)

In [ ]:
#3.kma classification
import subprocess
from pathlib import Path

input_dir = Path("/home/marcos/colaboracao/paulo_de_melo/RICHARDT-GAMA-LANDGRAF_results/RICHARDT-GAMA-LANDGRAF_results/downstream/3_filtered")
kma_out = Path("/home/marcos/colaboracao/paulo_de_melo/RICHARDT-GAMA-LANDGRAF_results/RICHARDT-GAMA-LANDGRAF_results/downstream/4_kma_taxa/silva")
kma_out.mkdir(parents=True, exist_ok=True)

# Replace with your actual KMA database path
kma_db = "/home/marcos/colaboracao/paulo_de_melo/RICHARDT-GAMA-LANDGRAF_results/RICHARDT-GAMA-LANDGRAF_results/downstream/kma_silva_db/silva_db"

fastq_files = [f for f in input_dir.iterdir() if f.name.endswith('.fastq')]

print(f"Starting KMA classification for {len(fastq_files)} samples...")

for file in fastq_files:
    sample_name = file.name.split('_filtered')[0]
    out_prefix = kma_out / sample_name
    
    cmd = [
        "conda", "run", "-n", "16s-nanopore",
        "kma",
        "-i", str(file),
        "-o", str(out_prefix),
        "-t_db", kma_db,
        "-bcNano",         # Optimizes for Nanopore errors
        "-ont", # Optimizes for ONT reads
        "-ID", "95.0",   # Reliability: Minimum 99% identity, indentification for species-level classification; but you can adjust for 97% indentification for genus-level classification
        "-bc", "0.7",    # Reliability: 99% statistical confidence
        "-t", "4"
    ]
    
    print(f"Running KMA for {sample_name}...")
    
    try:
        subprocess.run(cmd, check=True)
    except subprocess.CalledProcessError:
        print(f"[ERROR] Failed to classify {sample_name}")

print("\nKMA classification finished.")

Starting KMA classification for 12 samples...
Running KMA for FAT-1...


# Merge KMA results

In [8]:
import pandas as pd
from pathlib import Path

# 1. Define the directory where KMA saved the results
kma_dir = Path("/home/marcos/colaboracao/paulo_de_melo/RICHARDT-GAMA-LANDGRAF_results/RICHARDT-GAMA-LANDGRAF_results/downstream/4_kma_taxa/silva")

# 2. Find all KMA result files (.res)
res_files = list(kma_dir.glob("*.res"))
print(f"Found {len(res_files)} KMA result files. Merging them now...")

df_list = []

# 3. Read each file and extract the taxonomy and actual READ COUNTS
for file in res_files:
    sample_name = file.stem  # Gets the sample name without the .res extension
    
    # Read the tab-separated KMA output
    df = pd.read_csv(file, sep='\t')
    
    # Automatically detect the correct column for read counts
    if 'readCount' in df.columns:
        count_col = 'readCount'
    elif 'Expected' in df.columns:
        count_col = 'Expected'
    else:
        print(f"Warning: Could not find 'readCount' or 'Expected' in {file.name}")
        count_col = df.columns[2] # Fallback to the 3rd column
        
    # Keep only the Taxonomy Name (#Template) and the Read Count
    df = df[['#Template', count_col]].copy()
    df = df.rename(columns={count_col: sample_name})
    df.set_index('#Template', inplace=True)
    
    # KMA 'Expected' can have decimals (due to multi-mapping), so we round it to integers
    df = df.round(0).astype(int)
    
    df_list.append(df)

# 4. Combine all samples into a single matrix and fill missing values with 0
otu_matrix = pd.concat(df_list, axis=1).fillna(0).astype(int)

# 5. Save the final matrix
output_file = kma_dir / "otu_abundance_matrix.csv"
otu_matrix.to_csv(output_file)

print(f"\nMatrix successfully saved to:\n{output_file}")

Found 12 KMA result files. Merging them now...

Matrix successfully saved to:
/home/marcos/colaboracao/paulo_de_melo/RICHARDT-GAMA-LANDGRAF_results/RICHARDT-GAMA-LANDGRAF_results/downstream/4_kma_taxa/silva/otu_abundance_matrix.csv


1. kma classification

In [5]:
import pandas as pd
from pathlib import Path

# =========================================================
# 1. Paths
# =========================================================

matrix_dir = Path(
    "/home/marcos/colaboracao/paulo_de_melo/"
    "RICHARDT-GAMA-LANDGRAF_results/"
    "RICHARDT-GAMA-LANDGRAF_results/"
    "downstream/4_kma_taxa/silva"
)

raw_matrix_path = matrix_dir / "otu_abundance_matrix.csv"
clean_matrix_path = matrix_dir / "otu_abundance_matrix_CLEAN.csv"

tax_db_path = Path(
    "/home/marcos/colaboracao/paulo_de_melo/"
    "RICHARDT-GAMA-LANDGRAF_results/"
    "RICHARDT-GAMA-LANDGRAF_results/"
    "downstream/silva_database/silva_taxonomy.csv"
)

# =========================================================
# 2. Load KMA abundance matrix
# =========================================================

print("Loading KMA abundance matrix...")

df = pd.read_csv(raw_matrix_path)

# Extract tax_id from #Template
df["tax_id"] = (
    df["#Template"]
    .astype(str)
    .str.split(":")
    .str[0]
)

# Preserve original KMA template
df = df.rename(columns={"#Template": "raw_taxonomy"})

print(f"KMA entries: {len(df)}")
print(f"Unique tax IDs: {df['tax_id'].nunique()}")

# =========================================================
# 3. Load SILVA taxonomy manually
#    This handles malformed lines safely
# =========================================================

print("\nLoading SILVA taxonomy database...")

expected_fields = 8
taxonomy_rows = []
bad_lines = []

with open(tax_db_path, "r", encoding="utf-8", errors="replace") as f:

    for line_number, line in enumerate(f, start=1):

        line = line.rstrip("\n\r")

        fields = line.split(",")

        if len(fields) == expected_fields:

            taxonomy_rows.append(fields)

        elif len(fields) > expected_fields:

            # Keep first 7 taxonomy fields
            # Merge everything else into species
            fixed_fields = fields[:7] + [",".join(fields[7:])]

            taxonomy_rows.append(fixed_fields)

            bad_lines.append(
                (line_number, len(fields), line)
            )

        else:

            bad_lines.append(
                (line_number, len(fields), line)
            )

# =========================================================
# 4. Create taxonomy dataframe
# =========================================================

tax_columns = [
    "tax_id",
    "superkingdom",
    "phylum",
    "class",
    "order",
    "family",
    "genus",
    "species"
]

tax_db = pd.DataFrame(
    taxonomy_rows,
    columns=tax_columns
)

# Ensure tax_id is string
tax_db["tax_id"] = tax_db["tax_id"].astype(str)

print(f"SILVA taxonomy entries: {len(tax_db)}")
print(f"Unique SILVA tax IDs: {tax_db['tax_id'].nunique()}")

# =========================================================
# 5. Report malformed lines
# =========================================================

print("\nMalformed lines detected:")

if bad_lines:

    for line_number, n_fields, line in bad_lines[:20]:
        print(
            f"Line {line_number}: "
            f"{n_fields} fields"
        )
        print(line)

    if len(bad_lines) > 20:
        print(
            f"... and {len(bad_lines) - 20} additional lines."
        )

else:
    print("None.")

# =========================================================
# 6. Check tax_id matching
# =========================================================

matched = df["tax_id"].isin(tax_db["tax_id"])

print("\nTaxonomy matching:")
print(f"Matched: {matched.sum()}")
print(f"Not matched: {(~matched).sum()}")
print(f"Match rate: {matched.mean() * 100:.2f}%")

# Show unmatched IDs
if (~matched).sum() > 0:

    print("\nFirst unmatched tax IDs:")

    print(
        df.loc[~matched, "tax_id"]
        .drop_duplicates()
        .head(20)
        .to_string(index=False)
    )

# =========================================================
# 7. Merge abundance + taxonomy
# =========================================================

df_merged = pd.merge(
    df,
    tax_db,
    on="tax_id",
    how="left"
)

# =========================================================
# 8. Select final columns
# =========================================================

sample_cols = [
    col for col in df.columns
    if col not in ["tax_id", "raw_taxonomy"]
]

final_cols = tax_columns + sample_cols

df_clean = df_merged[final_cols]

# =========================================================
# 9. Save
# =========================================================

df_clean.to_csv(
    clean_matrix_path,
    index=False
)

print("\n========================================")
print("SUCCESS")
print("========================================")
print(f"Clean matrix saved to:")
print(clean_matrix_path)

print(f"\nFinal dimensions: {df_clean.shape}")

Loading KMA abundance matrix...
KMA entries: 40
Unique tax IDs: 40

Loading SILVA taxonomy database...
SILVA taxonomy entries: 431167
Unique SILVA tax IDs: 431167

Malformed lines detected:
Line 36145: 9 fields
LN867124.1.1427,Bacteria,Bacillota,Bacilli,Bacillales,Bacillaceae,Oceanobacillus,Ornithinibacillus sp. 1011MAR1A45,1
Line 49467: 9 fields
FN377734.1.1516,Bacteria,Actinomycetota,Actinobacteria,Micrococcales,Micrococcaceae,Paeniglutamicibacter,Arthrobacter sp. SH-93,5B
Line 62720: 9 fields
AY548757.1.1422,Bacteria,Pseudomonadota,Gammaproteobacteria,Burkholderiales,Rhodocyclaceae,Thauera,Thauera sp. Cin3,4
Line 127141: 9 fields
AF068011.1.1395,Bacteria,Pseudomonadota,Gammaproteobacteria,Burkholderiales,Burkholderiaceae,Burkholderia-Caballeronia-Paraburkholderia,Burkholderia sp. VUN 10,013
Line 192128: 10 fields
CP006602.296940.298481,Bacteria,Pseudomonadota,Gammaproteobacteria,Enterobacterales,Enterobacteriaceae,Salmonella,Salmonella enterica subsp. enterica serovar 4,[5],12:i:- s

## Reads annotation

In [6]:
import pandas as pd
from pathlib import Path

# 1. Define the path to your clean abundance matrix
clean_matrix_path = Path("/home/marcos/colaboracao/paulo_de_melo/RICHARDT-GAMA-LANDGRAF_results/RICHARDT-GAMA-LANDGRAF_results/downstream/4_kma_taxa/silva/otu_abundance_matrix_CLEAN.csv")

# 2. Load the matrix
df_clean = pd.read_csv(clean_matrix_path)

# 3. Separate taxonomy columns from sample columns
tax_cols = ['tax_id', 'superkingdom', 'phylum', 'class', 'order', 'family', 'genus', 'species', 'raw_taxonomy']
# Using sorted() to keep the output in alphabetical order (CTL-1, CTL-2, etc.)
sample_cols = sorted([col for col in df_clean.columns if col not in tax_cols])

print("\n--- Annotated Abundance per Sample ---")
print("* Note: Depending on your KMA settings, these numbers might represent alignment depth or score rather than absolute read counts.\n")

# 4. Calculate the sum for each sample
classified_counts = df_clean[sample_cols].sum()

# 5. Print the results
for sample in sample_cols:
    count = classified_counts[sample]
    print(f"  {sample}: {int(count):,}")
    
print("-" * 40)
print(f"  TOTAL OVERALL: {int(classified_counts.sum()):,}")
print("-" * 40)




--- Annotated Abundance per Sample ---
* Note: Depending on your KMA settings, these numbers might represent alignment depth or score rather than absolute read counts.

  CTL-1: 58
  CTL-2: 4
  CTL-3: 16
  DES-1: 25
  DES-2: 9
  DES-3: 107
  FAT-1: 53
  FAT-2: 32
  FAT-3: 29
  MOT-1: 49
  MOT-2: 123
  MOT-3: 53
----------------------------------------
  TOTAL OVERALL: 558
----------------------------------------


Counts reads of taxonomy classification

In [6]:
import pandas as pd
from pathlib import Path

# 1. Define o caminho da matriz limpa gerada no passo anterior
clean_matrix_path = Path("/home/marcos/colaboracao/paulo_de_melo/RICHARDT-GAMA-LANDGRAF_results/RICHARDT-GAMA-LANDGRAF_results/downstream/4_kma_taxa/ncbi/otu_abundance_matrix_CLEAN.csv")

# 2. Carrega a matriz
df_clean = pd.read_csv(clean_matrix_path)

# 3. Separa as colunas de taxonomia das colunas de amostras
tax_cols = ['tax_id', 'superkingdom', 'phylum', 'class', 'order', 'family', 'genus', 'species', 'raw_taxonomy']
sample_cols = [col for col in df_clean.columns if col not in tax_cols]

print("--- Classified Reads per Sample ---\n")

# 4. Calcula a soma de reads classificados para cada amostra
classified_counts = df_clean[sample_cols].sum()

# 5. Imprime os resultados formatados
for sample, count in classified_counts.items():
    print(f"  {sample}: {int(count):,} classified reads")
    
print("-" * 40)
print(f"  TOTAL OVERALL: {int(classified_counts.sum()):,} classified reads")
print("-" * 40)

--- Classified Reads per Sample ---

  MOT-1: 675,101 classified reads
  DES-3: 3,935,414 classified reads
  CTL-3: 933,738 classified reads
  MOT-3: 1,819,601 classified reads
  FAT-3: 1,264,026 classified reads
  CTL-2: 344,758 classified reads
  FAT-2: 1,286,955 classified reads
  MOT-2: 2,256,304 classified reads
  FAT-1: 1,510,857 classified reads
  DES-1: 1,664,400 classified reads
  DES-2: 477,565 classified reads
  CTL-1: 2,168,672 classified reads
----------------------------------------
  TOTAL OVERALL: 18,337,391 classified reads
----------------------------------------


## Creating different tables for analysis

In [7]:
import pandas as pd
import skbio
import os

# 1. Define paths
matrix_path = '/home/marcos/colaboracao/paulo_de_melo/RICHARDT-GAMA-LANDGRAF_results/RICHARDT-GAMA-LANDGRAF_results/downstream/4_kma_taxa/ncbi/otu_abundance_matrix_CLEAN.csv'
meta_path = '/home/marcos/colaboracao/paulo_de_melo/RICHARDT-GAMA-LANDGRAF_results/RICHARDT-GAMA-LANDGRAF_results/fastqs/meta_date_16s.csv'

# 2. Load OTU matrix and isolate taxonomy
otu_df = pd.read_csv(matrix_path)
tax_cols = ['tax_id', 'superkingdom', 'phylum', 'class', 'order', 'family', 'genus', 'species', 'raw_taxonomy']

# 3. Prepare Raw Counts (Genus level) safely
# Get only the sample columns dynamically (everything that is not taxonomy)
sample_cols = [col for col in otu_df.columns if col not in tax_cols]

# Keep only the genus and the samples, then sum by genus
counts_df = otu_df[['genus'] + sample_cols].copy()
counts_df = counts_df.groupby('genus').sum().T.astype(int)

# 4. Load Metadata and align samples
meta_df = pd.read_csv(meta_path).set_index('sample_title')
common_samples = counts_df.index.intersection(meta_df.index)
counts_df = counts_df.loc[common_samples]
meta_df = meta_df.loc[common_samples]

print("=== SETUP COMPLETE ===")
print(f"Samples ready: {counts_df.shape[0]}")
print(f"Unique Genera: {counts_df.shape[1]}\n")

# Verification of total reads per sample
sample_totals = counts_df.sum(axis=1).sort_values()
print("TOTAL READS PER SAMPLE (Look at this list to choose your depth!):")
print(sample_totals)
print("="*40 + "\n")

# ============================================================
# 5. Rarefy counts for Alpha Diversity
# ============================================================
# -> CHANGE THIS VALUE based on the list printed above! <-
rarefaction_depth =  1000

valid_samples = sample_totals[sample_totals >= rarefaction_depth].index
dropped_samples = set(counts_df.index) - set(valid_samples)

if dropped_samples:
    print(f"[WARNING] Samples dropped (less than {rarefaction_depth} reads): {dropped_samples}\n")

# Keep only valid samples
counts_df_valid = counts_df.loc[valid_samples]

# Apply rarefaction
counts_rarefied = counts_df_valid.apply(
    lambda x: skbio.stats.subsample_counts(x.values, rarefaction_depth), 
    axis=1, 
    result_type='broadcast'
)
counts_rarefied.columns = counts_df_valid.columns

print(f"Rarefied matrix created (Depth: {rarefaction_depth:,} reads/sample).")

# ============================================================
# Check reads after rarefaction
# ============================================================

rarefied_totals = counts_rarefied.sum(axis=1)
print("\nTotal reads per sample AFTER rarefaction:")
print(rarefied_totals)

total_overall = counts_rarefied.values.sum()
print(f"\nTotal overall reads kept in the dataset: {total_overall:,}")
print(f"Total samples retained: {len(counts_rarefied)}")

=== SETUP COMPLETE ===
Samples ready: 12
Unique Genera: 4

TOTAL READS PER SAMPLE (Look at this list to choose your depth!):
FAT-1      82888
CTL-3     121192
CTL-2     249555
FAT-2     368815
MOT-3     415962
DES-2     450932
CTL-1     478707
FAT-3     542353
DES-1     601579
DES-3     620865
MOT-1     667628
MOT-2    1550420
dtype: int64

Rarefied matrix created (Depth: 1,000 reads/sample).

Total reads per sample AFTER rarefaction:
FAT-1    1000
CTL-3    1000
CTL-2    1000
FAT-2    1000
MOT-3    1000
DES-2    1000
CTL-1    1000
FAT-3    1000
DES-1    1000
DES-3    1000
MOT-1    1000
MOT-2    1000
dtype: int64

Total overall reads kept in the dataset: 12,000
Total samples retained: 12


In [3]:
import os
import pandas as pd

# Fix Jupyter backend conflict
if 'MPLBACKEND' in os.environ:
    del os.environ['MPLBACKEND']

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

# 1. Prepare data comparing Before and After Rarefaction
compare_df = pd.DataFrame({
    'Original Reads': sample_totals
})

compare_df['After Rarefaction'] = counts_rarefied.sum(axis=1)
compare_df = compare_df.fillna(0)
compare_df = compare_df.sort_values('Original Reads', ascending=False)

# 2. Plotting
print("Generating rarefaction impact chart with labels...")
fig, ax = plt.subplots(figsize=(14, 8)) # Aumentei um pouco a altura para caber os textos

# Plot Original Reads (shows the discarded portion)
ax.bar(
    compare_df.index, 
    compare_df['Original Reads'], 
    color='#B8B8D8', 
    label='Original Reads (Discarded portion)', 
    width=0.8
)

# Plot Rarefied Reads (shows the retained portion)
ax.bar(
    compare_df.index, 
    compare_df['After Rarefaction'], 
    color='#6A5ACD', 
    label='Retained Reads (Rarefied)', 
    width=0.8
)

# Add a dashed line showing the threshold
ax.axhline(
    y=rarefaction_depth, 
    color='#ff7f0e', 
    linestyle='--', 
    linewidth=2, 
    label=f'Cutoff Threshold ({rarefaction_depth:,})'
)

# [NOVIDADE] Adicionando os textos (Retidas / Originais) no topo de cada barra
for i, (index, row) in enumerate(compare_df.iterrows()):
    original = int(row['Original Reads'])
    retained = int(row['After Rarefaction'])
    
    # Formato do texto: sobraram/tinham
    label_text = f"{retained}/{original}"
    
    ax.text(
        x=i, 
        y=original + (original * 0.02), # Posiciona 2% acima do topo da barra cinza
        s=label_text, 
        ha='center', 
        va='bottom',
        rotation=90,     # Rotaciona para evitar sobreposição
        fontsize=9,
        color='black'
    )

# 3. Formatting
ax.set_title("Impact of Rarefaction on Sample Reads", fontsize=15, pad=15)
ax.set_xlabel("Samples", fontsize=12)
ax.set_ylabel("Number of Reads", fontsize=12)

# Ajusta o limite superior do eixo Y para criar espaço para os textos rotacionados
max_reads = compare_df['Original Reads'].max()
ax.set_ylim(0, max_reads * 1.35) 

plt.setp(ax.get_xticklabels(), rotation=45, ha='right')
ax.ticklabel_format(style='plain', axis='y')
ax.grid(axis='y', linestyle=':', alpha=0.7)
ax.legend(loc="upper right", frameon=True)

plt.tight_layout()

# 4. Save Image
out_image = "/home/marcos/colaboracao/paulo_de_melo/RICHARDT-GAMA-LANDGRAF_results/RICHARDT-GAMA-LANDGRAF_results/downstream/4_kma_taxa/ncbi/rarefaction_impact_16s.png"
plt.savefig(out_image, dpi=300)
print(f"[SUCCESS] Chart saved to: {out_image}")

Generating rarefaction impact chart with labels...
[SUCCESS] Chart saved to: /home/marcos/colaboracao/paulo_de_melo/RICHARDT-GAMA-LANDGRAF_results/RICHARDT-GAMA-LANDGRAF_results/downstream/4_kma_taxa/ncbi/rarefaction_impact_16s.png


In [5]:
import os

# 1. Define output directory
out_dir = '/home/marcos/colaboracao/paulo_de_melo/RICHARDT-GAMA-LANDGRAF_results/RICHARDT-GAMA-LANDGRAF_results/downstream/5_statistics/kda'
os.makedirs(out_dir, exist_ok=True)

# 2. Filter metadata to match valid rarefied samples
meta_df_valid = meta_df.loc[valid_samples]

# 3. Save filtered metadata
meta_out_path = os.path.join(out_dir, 'metadata_filtered.csv')
meta_df_valid.to_csv(meta_out_path)

# 4. Save rarefied genus abundance counts
counts_out_path = os.path.join(out_dir, 'rarefied_genus_counts.csv')
counts_rarefied.to_csv(counts_out_path)

print(f"Tables successfully saved to:\n{out_dir}")

Tables successfully saved to:
/home/marcos/colaboracao/paulo_de_melo/RICHARDT-GAMA-LANDGRAF_results/RICHARDT-GAMA-LANDGRAF_results/downstream/5_statistics/kda


## Amcombc.r

In [ ]:
!conda create -n env_ancombc -c conda-forge -c bioconda bioconductor-ancombc bioconductor-phyloseq bioconductor-microbiome r-tidyverse -y
!conda activate env_ancombc